# ICA Pipeline

Пайплайн из трёх шагов:
1. **Шаг 1** — подобрать ICA для всех субъектов и сохранить `-ica.fif`
2. **Шаг 2** — визуализировать компоненты и заполнить `BAD_COMPONENTS`
3. **Шаг 3** — применить ICA, нарезать эпохи вокруг меток, сохранить

In [ ]:
import matplotlib
import matplotlib.axes

# Фикс совместимости MNE + matplotlib >= 3.8
if not hasattr(matplotlib.axes.Axes, '_axes_class'):
    matplotlib.axes.Axes._axes_class = matplotlib.axes.Axes

import mne
import numpy as np
from pathlib import Path

mne.set_log_level('WARNING')

BASE = Path(r'D:\Inner Speech Dataset\Inner Speech Dataset')
EMG_CHANNELS = ['B1+', 'B1-', 'B2+', 'B2-', 'EMG1', 'EMG2', 'Pharing1', 'Pharing2']

# ── Настройки ICA ─────────────────────────────────────────────
N_COMPONENTS = 20
RANDOM_STATE = 42
ICA_HP       = 1.0
ICA_LP       = 100.0

# ── Настройки нарезки эпох ────────────────────────────────────
WORD_EPOCH_DUR   = 1.0   # с — типичный субъект: 1 эпоха на метку
REPEAT_WINDOW_S  = 12.0  # с — атипичный: окно после метки → 1с нарезка
BG_WINDOW_S      = 30.0  # с — фон go/gz: окно после метки → 1с нарезка
ATYPICAL_GAP_THR = 7.0   # с: медиана интервалов > порога → атипичный субъект
BG_LABELS        = {'go', 'gz'}  # метки фона: глаза открыты / закрыты

# Ручной override (ключ 'язык/субъект'): True = атипичный
ATYPICAL_OVERRIDE: dict = {
    # 'Russian/sub5': True,
}

LANGUAGES = ['Russian', 'Spanish']

---
## Шаг 1 — Подобрать ICA и сохранить `-ica.fif`

Запускается один раз. Сохраняет ICA-объект рядом с данными субъекта.

In [ ]:
# Диагностика: каналы и ранг данных по каждому субъекту
for language in LANGUAGES:
    for edf in sorted((BASE / language).rglob('*.edf')):
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)
        drop = [ch for ch in EMG_CHANNELS if ch in raw.ch_names]
        raw.drop_channels(drop)
        rank = mne.compute_rank(raw, rank='info')['eeg']
        print(f'[{language}] {subject}: {len(raw.ch_names)} EEG каналов, ранг={rank}')

In [ ]:
def fit_ica_all(languages=LANGUAGES, overwrite=False):
    for language in languages:
        edf_files = sorted((BASE / language).rglob('*.edf'))
        for edf in edf_files:
            subject  = edf.parent.name
            ica_path = edf.parent / f'{subject}-ica.fif'

            if ica_path.exists() and not overwrite:
                print(f'[{language}] {subject} — уже есть, пропускаем')
                continue

            print(f'[{language}] {subject} — подбираем ICA...', end=' ')
            raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)

            drop = [ch for ch in EMG_CHANNELS if ch in raw.ch_names]
            raw.drop_channels(drop)

            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage, match_case=False, on_missing='ignore')

            raw_filt = raw.copy().filter(ICA_HP, ICA_LP, verbose=False)

            ica = mne.preprocessing.ICA(
                n_components=N_COMPONENTS,
                random_state=RANDOM_STATE,
                max_iter='auto',
            )
            ica.fit(raw_filt)
            ica.save(ica_path, overwrite=True)
            print(f'сохранено -> {ica_path.name} ({ica.n_components_} компонент)')


# Пересчитать только испанский датасет:
fit_ica_all(languages=['Spanish'], overwrite=True)

# Пересчитать всё:
# fit_ica_all(languages=LANGUAGES, overwrite=True)

---
## Шаг 2 — Визуализация компонент

Запускай поблочно для каждого субъекта.
После осмотра заполни `BAD_COMPONENTS` в следующей ячейке.

In [ ]:
%matplotlib qt

# Выбери субъекта для осмотра
INSPECT_LANGUAGE = 'Russian'
INSPECT_SUBJECT  = 'sub1'

edf_path = BASE / INSPECT_LANGUAGE / INSPECT_SUBJECT / f'{INSPECT_SUBJECT}.edf'
ica_path = edf_path.parent / f'{INSPECT_SUBJECT}-ica.fif'

raw_inspect = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
drop = [ch for ch in EMG_CHANNELS if ch in raw_inspect.ch_names]
raw_inspect.drop_channels(drop)
montage = mne.channels.make_standard_montage('standard_1020')
raw_inspect.set_montage(montage, match_case=False, on_missing='ignore')
raw_filt_inspect = raw_inspect.copy().filter(ICA_HP, ICA_LP, verbose=False)

ica_inspect = mne.preprocessing.read_ica(ica_path)
n_comp = ica_inspect.n_components_
print(f'Компонент в ICA: {n_comp}')

ica_inspect.plot_components(inst=raw_filt_inspect, picks=range(n_comp))

In [ ]:
%matplotlib qt
ica_inspect.plot_sources(raw_filt_inspect, block=True)

In [ ]:
# Детальный осмотр конкретной компоненты
# ica_inspect.plot_properties(raw_filt_inspect, picks=[0, 1])

---
## Шаг 3 — Задать плохие компоненты и пересобрать fif

Заполни `BAD_COMPONENTS` после осмотра.

**Логика нарезки эпох:**
- **Типичный субъект** (одно произнесение на метку): 1 эпоха × 1 с на каждую метку слова.
- **Атипичный субъект** (непрерывные повторения): окно `REPEAT_WINDOW_S` с после метки → 1с нарезка.
  Тип определяется автоматически по медиане интервалов между метками (`ATYPICAL_GAP_THR`)
  или вручную через `ATYPICAL_OVERRIDE`.
- **Метки go / gz** (фон): окно `BG_WINDOW_S` = 30 с → 1с нарезка (все субъекты).

In [ ]:
# ── Заполни после осмотра ─────────────────────────────────────
BAD_COMPONENTS = {
    # 'Russian/sub1':  [0, 1],
    # 'Russian/sub2':  [2],
    # 'Spanish/sub0':  [0, 3],
}

In [ ]:
def _detect_subject_type(raw, key):
    """Автоопределение типа субъекта: typical / atypical."""
    if key in ATYPICAL_OVERRIDE:
        return 'atypical' if ATYPICAL_OVERRIDE[key] else 'typical'
    word_onsets = np.array([
        onset for onset, desc in zip(raw.annotations.onset, raw.annotations.description)
        if desc.lower() not in BG_LABELS
    ])
    if len(word_onsets) < 3:
        return 'atypical'
    gaps = np.diff(np.sort(word_onsets))
    return 'atypical' if float(np.median(gaps)) > ATYPICAL_GAP_THR else 'typical'


def _sliding_events(start_sample, window_s, sfreq, n_data, event_code):
    """Создать события через каждую секунду внутри окна window_s."""
    step = int(round(sfreq))
    return [
        [start_sample + i * step, 0, event_code]
        for i in range(int(window_s))
        if start_sample + i * step + step <= n_data
    ]


def apply_ica_and_epoch(languages=LANGUAGES):
    for language in languages:
        for edf in sorted((BASE / language).rglob('*.edf')):
            subject  = edf.parent.name
            key      = f'{language}/{subject}'
            ica_path = edf.parent / f'{subject}-ica.fif'
            out_path = edf.parent / f'{subject}-clean-epo.fif'

            if not ica_path.exists():
                print(f'[{key}] нет ICA-файла, пропускаем')
                continue

            # Загрузка и монтаж
            raw = mne.io.read_raw_edf(edf, preload=True, verbose=False)
            drop = [ch for ch in EMG_CHANNELS if ch in raw.ch_names]
            raw.drop_channels(drop)
            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage, match_case=False, on_missing='ignore')

            # Применить ICA
            ica = mne.preprocessing.read_ica(ica_path)
            ica.exclude = BAD_COMPONENTS.get(key, [])
            ica.apply(raw)

            # Тип субъекта
            stype = _detect_subject_type(raw, key)
            print(f'[{key}] тип={stype}, exclude={ica.exclude}', end=' ')

            # Собрать события вручную
            sfreq   = raw.info['sfreq']
            n_data  = len(raw.times)
            descs   = sorted(set(raw.annotations.description))
            event_id = {desc: i + 1 for i, desc in enumerate(descs)}

            all_events = []
            for onset, desc in zip(raw.annotations.onset, raw.annotations.description):
                code  = event_id[desc]
                start = int(round(onset * sfreq))
                if desc.lower() in BG_LABELS:
                    all_events += _sliding_events(start, BG_WINDOW_S, sfreq, n_data, code)
                elif stype == 'atypical':
                    all_events += _sliding_events(start, REPEAT_WINDOW_S, sfreq, n_data, code)
                else:
                    if start + int(sfreq) <= n_data:
                        all_events.append([start, 0, code])

            if not all_events:
                print('→ нет событий, пропускаем')
                continue

            events = np.array(all_events, dtype=int)
            events = events[events[:, 0].argsort()]

            # tmax = 1 с − 1 сэмпл → ровно sfreq точек на эпоху
            tmax = WORD_EPOCH_DUR - 1.0 / sfreq
            epochs = mne.Epochs(
                raw, events, event_id=event_id,
                tmin=0.0, tmax=tmax,
                baseline=None,
                preload=True, verbose=False,
                event_repeated='drop',
            )
            epochs.save(out_path, overwrite=True)
            print(f'→ {out_path.name}  ({len(epochs)} эпох)')


apply_ica_and_epoch()